In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1EBnaG11Xe83RcBkM3djhgyJrNlvS6IFh", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Notebook Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_notebook_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# Entity Memory: Structured Fact Extraction and Storage

*Part 3 of the Vizuara series on Memory Architectures for LLM Applications*

**Estimated time: 55 minutes**

Summary memory compresses old conversations into a paragraph of prose. This preserves themes and general context, but specific details — the exact budget number, the precise deadline, a team member's role — often vanish in the compression. What if we could surgically extract those specific facts and store them in a structured dictionary? That is entity memory. Instead of compressing everything into a summary, we identify key entities and facts after each turn and store them as structured key-value pairs. When the user asks a question, we look up the relevant entities and inject them into the prompt. In this notebook, we will build a complete entity extraction and storage system, test its retrieval accuracy, and understand when structured memory beats unstructured summaries.

In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are a doctor with hundreds of patients. When a patient walks in, you do not read through every page of their medical history. Instead, you pull up their **chart** — a structured record with their name, age, allergies, medications, conditions, and recent visits. Each piece of information is labeled, organized, and instantly accessible.

Entity memory does the same thing for an LLM. Instead of dumping raw conversation text into the prompt, we maintain a structured record:

```
{
  "user_name": "Raj",
  "budget": "$5,000",
  "deadline": "March 15",
  "team_size": 4,
  "tech_stack": ["Python", "FastAPI", "PostgreSQL"],
  "allergies": "none mentioned",
  ...
}
```

When the user asks "What is my budget?", we look up the `budget` key and inject "$5,000" into the prompt. No summary needed, no compression, no information loss. The exact number survives.

This approach has three key advantages over summary memory:
1. **Precision**: Specific facts are stored exactly as stated, not compressed
2. **Efficiency**: We inject only the facts relevant to the current query
3. **Updateability**: When a fact changes ("Actually, the budget is now $8,000"), we simply update the value

In this notebook, we will:
- Build an **entity extractor** that identifies facts from conversation turns
- Implement a **structured entity store** with typed categories
- Test **retrieval accuracy** — can the system find the right facts?
- Compare entity memory against summary memory on fact recall
- Build a **relevance-based entity selector** that picks which facts to include

Let us begin.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

### What Counts as an "Entity"?

In the NLP world, "entity" typically means named entities — people, organizations, locations. But for memory purposes, we expand this to include any **specific, referenceable fact**:

| Category | Examples | Why It Matters |
|----------|----------|----------------|
| **Names** | Raj, Sarah, Priya | Personalization |
| **Numbers** | $5,000, 4 people, 128K tokens | Precision |
| **Dates** | March 15, last Tuesday | Temporal grounding |
| **Preferences** | Prefers Python, hates meetings | Personalization |
| **Technical** | Uses FastAPI, PostgreSQL | Context for answers |
| **Relationships** | Sarah does frontend, Alex does backend | Team awareness |

### Extraction vs Retrieval

Entity memory has two distinct phases:

1. **Extraction (write path):** After each conversation turn, scan the message for entities and store them. This happens on every turn.
2. **Retrieval (read path):** When building the prompt, select the entities most relevant to the current query. This happens before each LLM call.

The extraction step is the harder one. Natural language is messy. "I'm using Python" and "We decided on Python for the backend" both express the same entity (tech_stack: Python), but the surface forms are very different.

### The Relevance Problem

Not all stored entities are relevant to every query. If the user asks "What database should I use?", injecting their name, budget, and dietary preferences into the prompt is wasteful. We need a way to select which entities to include.

The simplest approach: keyword matching. If the query contains "database", retrieve entities related to databases. A more sophisticated approach: embed both the query and entity descriptions, then retrieve by semantic similarity. We will build both.

In [ ]:
#@title 🎧 Listen: The Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_the_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

### Token Cost of Entity Memory

Let $E$ be the set of stored entities, each with a description of $d_i$ tokens. Let $R(q, E) \subseteq E$ be the subset of entities relevant to query $q$. The token cost is:

$$C_{\text{entity}} = \sum_{e \in R(q, E)} d_e + t_{\text{recent}}$$

where $t_{\text{recent}}$ is the token cost of the recent verbatim messages.

**Worked example:** Suppose we have 20 entities averaging 30 tokens each ($20 \times 30 = 600$ tokens total). For a given query, only 5 are relevant ($5 \times 30 = 150$ tokens). With 1,000 tokens of recent messages:

$$C = 150 + 1{,}000 = 1{,}150 \text{ tokens}$$

Compare this to summary memory ($300 + 1{,}000 = 1{,}300$) — entity memory can actually be cheaper when only a few entities are relevant, and it preserves exact facts that the summary would lose.

### Extraction Precision and Recall

To evaluate our extractor, we use information retrieval metrics:

$$\text{Precision} = \frac{|\text{Correct extractions}|}{|\text{All extractions}|}$$

$$\text{Recall} = \frac{|\text{Correct extractions}|}{|\text{All true entities}|}$$

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Worked example:** A message contains 5 true entities. Our extractor finds 4 entities, 3 of which are correct (1 is a false positive like extracting "Great" as a name).

$$\text{Precision} = \frac{3}{4} = 75\% \qquad \text{Recall} = \frac{3}{5} = 60\% \qquad F_1 = 2 \cdot \frac{0.75 \times 0.60}{0.75 + 0.60} = 66.7\%$$

This tells us the extractor is decent but misses some entities and occasionally hallucinates. In production, using an LLM for extraction dramatically improves both precision and recall.

### Retrieval Relevance Scoring

For keyword-based retrieval, we score each entity by the number of query keywords that match:

$$\text{relevance}(q, e) = \frac{|\text{keywords}(q) \cap \text{keywords}(e)|}{|\text{keywords}(q)|}$$

**Worked example:** Query "What database should I use for the project?" has keywords {database, project}. Entity "tech_stack: PostgreSQL for the project database" has keywords {postgresql, project, database}. Overlap is {database, project}, so:

$$\text{relevance} = \frac{2}{2} = 100\%$$

This entity is highly relevant and should be included.

In [ ]:
#@title 🎧 Listen: Env Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_env_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Setting Up Our Environment

In [ ]:
# Install dependencies
!pip install -q tiktoken numpy matplotlib

In [ ]:
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional, Set
from collections import defaultdict
from dataclasses import dataclass, field
import re
import json
import time

ENCODER = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))

print("Environment ready!")

In [ ]:
#@title 🎧 Listen: Entity Data Structure
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_entity_data_structure.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 The Entity Data Structure

First, let us define what an entity looks like. Each entity has a category, a key (identifier), a value, and metadata about when and how it was extracted.

In [ ]:
@dataclass
class Entity:
    """A single extracted entity/fact."""
    category: str       # 'name', 'number', 'date', 'preference', 'technical', 'relationship'
    key: str            # Identifier, e.g., 'user_name', 'budget'
    value: str          # The actual value, e.g., 'Raj', '$5,000'
    source_turn: int    # Which conversation turn it came from
    confidence: float   # Extraction confidence (0-1)
    keywords: List[str] = field(default_factory=list)

    @property
    def tokens(self) -> int:
        return count_tokens(f"{self.key}: {self.value}")

    def __repr__(self):
        return f"Entity({self.category}/{self.key}={self.value})"

# Quick test
e = Entity(
    category="number", key="budget",
    value="$5,000", source_turn=1,
    confidence=0.95,
    keywords=["budget", "money", "cost", "5000"],
)
print(f"Entity: {e}")
print(f"Tokens: {e.tokens}")

In [ ]:
#@title 🎧 Listen: Entity Extractor
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_entity_extractor.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 The Entity Extractor

Now we build the extraction pipeline. This is a rule-based system that mimics what an LLM would do — identify patterns in text and extract structured entities.

In [ ]:
class EntityExtractor:
    """Rule-based entity extractor.

    Identifies names, numbers, dates, preferences,
    technical facts, and relationships from text.
    In production, you would replace this with an
    LLM call that outputs structured JSON.
    """

    def __init__(self):
        self.extraction_count = 0

    def extract(self, text: str, turn_number: int = 0) -> List[Entity]:
        """Extract entities from a text string."""
        self.extraction_count += 1
        entities = []

        lower = text.lower()

        # --- Name patterns ---
        name_patterns = [
            (r"my name is (\w+)", "user_name"),
            (r"i am (\w+)", "user_name"),
            (r"i'm (\w+)", "user_name"),
            (r"(\w+) (?:handles?|does|works on|is on) (\w+)",
             "team_member"),
        ]
        for pattern, key in name_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                if key == "team_member":
                    name = match.group(1)
                    role = match.group(2)
                    if name[0].isupper() and name not in (
                        "The", "This", "That", "What", "I"
                    ):
                        entities.append(Entity(
                            category="relationship",
                            key=f"team_{name.lower()}",
                            value=f"{name} works on {role}",
                            source_turn=turn_number,
                            confidence=0.8,
                            keywords=[name.lower(), role.lower(), "team"],
                        ))
                else:
                    name = match.group(1)
                    if name not in ("a", "an", "the", "not", "very"):
                        entities.append(Entity(
                            category="name",
                            key=key,
                            value=name,
                            source_turn=turn_number,
                            confidence=0.9,
                            keywords=[name.lower(), "name", "user"],
                        ))

        # --- Dollar amounts ---
        for match in re.finditer(
            r'\$[\d,]+(?:\.\d{2})?', text
        ):
            amount = match.group()
            entities.append(Entity(
                category="number",
                key="budget",
                value=amount,
                source_turn=turn_number,
                confidence=0.95,
                keywords=["budget", "money", "cost", "dollar",
                          amount.replace("$", "").replace(",", "")],
            ))

        # --- Number + unit patterns ---
        for match in re.finditer(
            r'(\d+)\s+(people|members|users|days|weeks|'
            r'months|hours|percent|%)', text
        ):
            num = match.group(1)
            unit = match.group(2)
            entities.append(Entity(
                category="number",
                key=f"count_{unit}",
                value=f"{num} {unit}",
                source_turn=turn_number,
                confidence=0.85,
                keywords=[unit, num, "count", "number"],
            ))

        # --- Date patterns ---
        for match in re.finditer(
            r'((?:January|February|March|April|May|June|'
            r'July|August|September|October|November|'
            r'December)\s+\d{1,2})', text
        ):
            date_str = match.group(1)
            entities.append(Entity(
                category="date",
                key="deadline",
                value=date_str,
                source_turn=turn_number,
                confidence=0.9,
                keywords=["deadline", "date", "due",
                          date_str.lower()],
            ))

        # --- Preference patterns ---
        pref_patterns = [
            (r"i (?:prefer|like|love|enjoy) (\w+(?:\s+\w+)?)",
             "preference"),
            (r"i (?:hate|dislike|avoid) (\w+(?:\s+\w+)?)",
             "dislike"),
        ]
        for pattern, ptype in pref_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                thing = match.group(1)
                if thing not in ("to", "a", "the", "it", "that"):
                    entities.append(Entity(
                        category="preference",
                        key=f"{ptype}_{thing.lower().replace(' ', '_')}",
                        value=f"{'Likes' if ptype == 'preference' else 'Dislikes'} {thing}",
                        source_turn=turn_number,
                        confidence=0.8,
                        keywords=[thing.lower(), ptype, "preference"],
                    ))

        # --- Technical patterns ---
        tech_keywords = {
            "python": "programming language",
            "fastapi": "web framework",
            "react": "frontend framework",
            "postgresql": "database",
            "postgres": "database",
            "pgvector": "vector extension",
            "docker": "containerization",
            "kubernetes": "orchestration",
            "oauth": "authentication",
            "jwt": "authentication",
            "faiss": "vector search",
            "redis": "caching",
            "mongodb": "database",
            "tensorflow": "ML framework",
            "pytorch": "ML framework",
            "jax": "ML framework",
        }
        for tech, desc in tech_keywords.items():
            if tech in lower:
                entities.append(Entity(
                    category="technical",
                    key=f"tech_{tech}",
                    value=f"{tech.capitalize()}: {desc}",
                    source_turn=turn_number,
                    confidence=0.9,
                    keywords=[tech, desc.split()[0].lower(),
                              "technology", "stack"],
                ))

        return entities

print("EntityExtractor defined!")

In [ ]:
#@title 🎧 Listen: Extractor Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_extractor_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us test the extractor on some sample messages:

In [ ]:
extractor = EntityExtractor()

test_messages = [
    "My name is Raj and I am building an e-commerce chatbot.",
    "The budget is $5,000 and the deadline is March 15.",
    "The team has 4 people: Sarah handles frontend, "
    "Alex does backend, Priya works on ML.",
    "I prefer Python over JavaScript. We are using "
    "FastAPI and PostgreSQL.",
    "I hate unnecessary meetings.",
]

print("Entity Extraction Results")
print("=" * 60)

all_entities = []
for i, msg in enumerate(test_messages):
    entities = extractor.extract(msg, turn_number=i + 1)
    all_entities.extend(entities)
    print(f"\nTurn {i+1}: \"{msg[:50]}...\"")
    if entities:
        for e in entities:
            print(f"  [{e.category:>12}] {e.key}: {e.value} "
                  f"(conf={e.confidence:.0%})")
    else:
        print("  (no entities extracted)")

print(f"\nTotal entities extracted: {len(all_entities)}")
print(f"Categories: {dict(defaultdict(int, {e.category: sum(1 for x in all_entities if x.category == e.category) for e in all_entities}))}")

In [ ]:
#@title 🎧 Listen: Entity Store
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_entity_store.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 The Entity Store

Now we build the storage layer. The entity store handles deduplication (if the user mentions their name twice), updates (if the budget changes), and retrieval by relevance.

In [ ]:
class EntityStore:
    """Structured entity storage with dedup, update, and retrieval.

    Stores entities in a dictionary keyed by entity key.
    Handles updates when the same key is extracted again
    (keeps the more recent or higher-confidence version).
    """

    def __init__(self):
        self.entities: Dict[str, Entity] = {}
        self.history: List[Tuple[str, Entity]] = []  # (action, entity)

    def upsert(self, entity: Entity):
        """Insert or update an entity.

        If the key already exists, update only if the new
        entity is more recent or has higher confidence.
        """
        key = entity.key
        if key in self.entities:
            existing = self.entities[key]
            # Update if newer or higher confidence
            if (entity.source_turn > existing.source_turn
                    or entity.confidence > existing.confidence):
                old_val = existing.value
                self.entities[key] = entity
                self.history.append(("update", entity))
                return f"Updated {key}: '{old_val}' -> '{entity.value}'"
            return f"Kept existing {key}: '{existing.value}'"
        else:
            self.entities[key] = entity
            self.history.append(("insert", entity))
            return f"Inserted {key}: '{entity.value}'"

    def get(self, key: str) -> Optional[Entity]:
        """Look up an entity by key."""
        return self.entities.get(key)

    def get_all(self) -> List[Entity]:
        """Return all stored entities."""
        return list(self.entities.values())

    def get_by_category(self, category: str) -> List[Entity]:
        """Return all entities in a category."""
        return [
            e for e in self.entities.values()
            if e.category == category
        ]

    def retrieve_relevant(
        self,
        query: str,
        top_k: int = 10,
    ) -> List[Tuple[Entity, float]]:
        """Retrieve entities relevant to a query using
        keyword matching.

        For each entity, compute:
          relevance = |query_words & entity_keywords| / |query_words|

        Returns (entity, relevance_score) pairs sorted
        by score descending.
        """
        query_words = set(query.lower().split())
        # Remove common stopwords
        stopwords = {
            "the", "a", "an", "is", "are", "was", "were",
            "what", "how", "can", "do", "did", "my", "your",
            "i", "me", "we", "you", "to", "for", "of", "in",
            "on", "at", "by", "and", "or", "but", "not",
            "this", "that", "it", "about", "with",
        }
        query_words -= stopwords

        if not query_words:
            return [(e, 0.5) for e in self.get_all()[:top_k]]

        scored = []
        for entity in self.entities.values():
            entity_words = set(
                kw.lower() for kw in entity.keywords
            )
            entity_words |= set(entity.value.lower().split())
            entity_words |= set(entity.key.lower().split("_"))

            overlap = query_words & entity_words
            score = len(overlap) / len(query_words)
            if score > 0:
                scored.append((entity, score))

        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def format_for_prompt(
        self,
        entities: List[Entity] = None,
    ) -> str:
        """Format entities as a string for injection into
        the LLM prompt."""
        if entities is None:
            entities = self.get_all()
        lines = []
        for e in entities:
            lines.append(f"- {e.key}: {e.value}")
        return "\n".join(lines)

    @property
    def size(self):
        return len(self.entities)

    @property
    def total_tokens(self):
        return sum(e.tokens for e in self.entities.values())

print("EntityStore defined!")

In [ ]:
#@title 🎧 Listen: Store Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_store_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us populate the store and test retrieval:

In [ ]:
store = EntityStore()

# Insert all extracted entities
for entity in all_entities:
    result = store.upsert(entity)
    print(f"  {result}")

print(f"\nStore size: {store.size} entities")
print(f"Total tokens: {store.total_tokens}")

# Test retrieval
print(f"\n--- Retrieval Test ---")
queries = [
    "What is the budget?",
    "Who is on the team?",
    "What tech stack are we using?",
    "When is the deadline?",
]

for query in queries:
    results = store.retrieve_relevant(query, top_k=3)
    print(f"\nQuery: \"{query}\"")
    for entity, score in results:
        print(f"  [{score:.2f}] {entity.key}: {entity.value}")

In [ ]:
#@title 🎧 Listen: Complete System
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_complete_system.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 The Complete Entity Memory System

Now let us wire everything together into a complete memory system that automatically extracts entities from every turn and retrieves them for prompt building.

In [ ]:
class EntityMemory:
    """Complete entity memory system.

    Automatically extracts entities from each conversation
    turn, stores them with deduplication, and retrieves
    relevant entities for prompt construction.
    """

    def __init__(
        self,
        recent_turns: int = 5,
        max_entity_tokens: int = 500,
        system_prompt: str = "",
    ):
        self.extractor = EntityExtractor()
        self.store = EntityStore()
        self.recent_turns = recent_turns
        self.max_entity_tokens = max_entity_tokens
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)
        self.all_messages: List[Dict] = []
        self.turn_count = 0

        # Tracking
        self.context_history: List[int] = []
        self.entity_count_history: List[int] = []
        self.extraction_log: List[Dict] = []

    def add_message(self, role: str, content: str):
        """Add a message and extract entities from it."""
        tokens = count_tokens(content)
        self.all_messages.append({
            "role": role, "content": content, "tokens": tokens,
        })

        if role == "user":
            self.turn_count += 1

        # Extract entities from this message
        entities = self.extractor.extract(
            content, turn_number=self.turn_count
        )
        for e in entities:
            self.store.upsert(e)

        self.extraction_log.append({
            "turn": self.turn_count,
            "role": role,
            "entities_extracted": len(entities),
            "store_size": self.store.size,
        })

        # Track context size
        ctx_tokens = self._compute_context_tokens("general query")
        self.context_history.append(ctx_tokens)
        self.entity_count_history.append(self.store.size)

    def build_context(self, query: str) -> str:
        """Build the full context for an LLM prompt.

        Components:
        1. System prompt
        2. Relevant entity facts
        3. Recent messages (verbatim)
        4. Current query
        """
        parts = []
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")

        # Retrieve relevant entities
        relevant = self.store.retrieve_relevant(query, top_k=15)

        # Filter to fit token budget
        selected = []
        entity_tokens = 0
        for entity, score in relevant:
            if entity_tokens + entity.tokens > self.max_entity_tokens:
                break
            selected.append(entity)
            entity_tokens += entity.tokens

        if selected:
            entity_text = self.store.format_for_prompt(selected)
            parts.append(f"Known facts about the user:\n{entity_text}")

        # Recent messages
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]
        if recent:
            recent_text = "\n".join(
                f"{m['role']}: {m['content']}" for m in recent
            )
            parts.append(f"Recent conversation:\n{recent_text}")

        return "\n\n".join(parts)

    def _compute_context_tokens(self, query: str) -> int:
        """Estimate context tokens for a query."""
        relevant = self.store.retrieve_relevant(query, top_k=15)
        entity_tokens = min(
            sum(e.tokens for e, _ in relevant),
            self.max_entity_tokens,
        )
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]
        recent_tokens = sum(m["tokens"] for m in recent)
        return self.system_tokens + entity_tokens + recent_tokens

    def get_stats(self) -> Dict:
        return {
            "total_messages": len(self.all_messages),
            "total_turns": self.turn_count,
            "entities_stored": self.store.size,
            "entity_tokens": self.store.total_tokens,
            "extraction_calls": self.extractor.extraction_count,
        }

print("EntityMemory system defined!")

In [ ]:
#@title 🎧 Listen: System Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_system_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us test the full system on a realistic conversation:

In [ ]:
em = EntityMemory(
    recent_turns=3,
    max_entity_tokens=500,
    system_prompt="You are a project planning assistant.",
)

conversation = [
    ("user", "My name is Raj. I am building an e-commerce chatbot."),
    ("assistant", "Hello Raj! What is your budget and timeline?"),
    ("user", "Budget is $5,000 and deadline is March 15. "
     "Team has 4 people."),
    ("assistant", "Got it! $5,000 budget, March 15 deadline, 4-person team."),
    ("user", "Sarah handles frontend with React. Alex does "
     "backend with FastAPI. Priya works on ML models."),
    ("assistant", "Great allocation. Python, React, and FastAPI."),
    ("user", "We need PostgreSQL with pgvector for embeddings."),
    ("assistant", "PostgreSQL with pgvector is an excellent choice."),
    ("user", "I prefer Docker for deployment. Also need OAuth2."),
    ("assistant", "Docker + OAuth2. I will plan around those."),
    ("user", "I hate meetings that could be emails."),
    ("assistant", "Understood — we will use async communication."),
]

for role, content in conversation:
    em.add_message(role, content)

stats = em.get_stats()
print(f"Conversation: {stats['total_turns']} turns")
print(f"Entities stored: {stats['entities_stored']}")
print(f"Entity tokens: {stats['entity_tokens']}")
print(f"\n--- All Stored Entities ---")
for e in em.store.get_all():
    print(f"  [{e.category:>12}] {e.key}: {e.value}")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Implement Entity Update with Contradiction Detection

When a user changes a fact ("Actually, the budget is now $8,000"), the entity store should detect the contradiction, update the value, and keep a history of changes. Implement the `SmartEntityStore` with contradiction detection.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class SmartEntityStore(EntityStore):
    """Entity store with contradiction detection and
    change history tracking.

    When an entity is updated with a conflicting value,
    this store records the contradiction and keeps a
    history of all changes for each key.

    Args:
        single_valued_keys: Set of keys that should only
            have one value (e.g., 'budget', 'deadline').
            Updating these with a different value is a
            contradiction.
    """

    def __init__(self):
        super().__init__()
        self.contradictions: List[Dict] = []
        self.change_history: Dict[str, List[Dict]] = defaultdict(list)
        self.single_valued = {
            "budget", "deadline", "user_name",
            "count_people", "count_members",
        }

    def upsert_with_detection(
        self, entity: Entity
    ) -> Dict:
        """Insert or update an entity with contradiction
        detection.

        Args:
            entity: The entity to insert/update

        Returns:
            Dict with keys:
                'action': 'insert', 'update', or 'no_change'
                'contradiction': True/False
                'old_value': previous value (if update)
                'new_value': new value
                'key': entity key

        Requirements:
            1. Check if key exists in self.entities
            2. If it exists AND key is in self.single_valued:
               a. Check if values are different (case-insensitive)
               b. If different: record contradiction in
                  self.contradictions with keys 'key',
                  'old_value', 'new_value', 'turn'
               c. Update the entity (new value wins)
               d. Record in change_history
            3. If it exists but NOT single-valued:
               just update normally
            4. If new key: insert normally
            5. Return the result dict

        Example:
            >>> store = SmartEntityStore()
            >>> store.upsert_with_detection(Entity(
            ...     'number', 'budget', '$5,000', 1, 0.9, []))
            {'action': 'insert', 'contradiction': False, ...}
            >>> store.upsert_with_detection(Entity(
            ...     'number', 'budget', '$8,000', 5, 0.9, []))
            {'action': 'update', 'contradiction': True,
             'old_value': '$5,000', 'new_value': '$8,000', ...}
        """
        # ---- YOUR CODE HERE ---- #

        result = {
            "action": "insert",
            "contradiction": False,
            "old_value": None,
            "new_value": entity.value,
            "key": entity.key,
        }

        # TODO: Check for existing entity
        # TODO: Detect contradiction if single-valued
        # TODO: Record in change_history
        # TODO: Perform the upsert

        return result

        # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Implement a Retrieval Accuracy Evaluator

Build a function that evaluates how accurately the entity memory system can answer factual questions about the conversation.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def evaluate_retrieval_accuracy(
    memory: EntityMemory,
    test_cases: List[Dict],
) -> Dict:
    """Evaluate entity retrieval accuracy on test cases.

    Args:
        memory: A populated EntityMemory instance
        test_cases: List of dicts, each with:
            - 'query': The question to ask
            - 'expected_key': The entity key that should
              be retrieved
            - 'expected_value_keywords': List of keywords
              that should appear in the retrieved value
            - 'category': Type of test ('name', 'number',
              'date', 'technical', 'preference')

    Returns:
        Dict with keys:
            'total_cases': number of test cases
            'correct': number where expected entity was
                in top-3 results AND value keywords matched
            'accuracy': correct / total_cases
            'per_category': dict mapping category to
                {'total': int, 'correct': int, 'accuracy': float}
            'failures': list of failed test case queries

    Steps:
        1. For each test case:
           a. Call memory.store.retrieve_relevant(query, top_k=5)
           b. Check if any result has key == expected_key
           c. If found, check if expected_value_keywords
              appear in the entity's value (case-insensitive)
           d. Record success or failure
        2. Compute per-category accuracy
        3. Return results dict

    Example:
        >>> cases = [
        ...   {'query': 'What is the budget?',
        ...    'expected_key': 'budget',
        ...    'expected_value_keywords': ['5,000'],
        ...    'category': 'number'},
        ... ]
        >>> result = evaluate_retrieval_accuracy(em, cases)
        >>> result['accuracy']
        1.0
    """
    # ---- YOUR CODE HERE ---- #

    results = {
        "total_cases": len(test_cases),
        "correct": 0,
        "accuracy": 0.0,
        "per_category": {},
        "failures": [],
    }

    # TODO: Iterate through test cases
    # TODO: Check retrieval results
    # TODO: Compute accuracy

    return results

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: Verify Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_verify_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 1

In [ ]:
# --- Test SmartEntityStore ---
smart_store = SmartEntityStore()

# Insert initial budget
r1 = smart_store.upsert_with_detection(Entity(
    "number", "budget", "$5,000", 1, 0.9,
    ["budget", "money", "5000"],
))
assert r1["action"] == "insert"
assert r1["contradiction"] == False
print(f"Insert: {r1}")

# Update budget (contradiction!)
r2 = smart_store.upsert_with_detection(Entity(
    "number", "budget", "$8,000", 5, 0.9,
    ["budget", "money", "8000"],
))
assert r2["action"] == "update"
assert r2["contradiction"] == True
assert r2["old_value"] == "$5,000"
assert r2["new_value"] == "$8,000"
print(f"Update: {r2}")

# Check contradiction was recorded
assert len(smart_store.contradictions) == 1
print(f"Contradiction: {smart_store.contradictions[0]}")

# Non-contradicting insert
r3 = smart_store.upsert_with_detection(Entity(
    "name", "user_name", "Raj", 1, 0.9, ["raj", "name"],
))
assert r3["action"] == "insert"
assert r3["contradiction"] == False
print(f"Insert name: {r3}")

# Multi-valued key (preferences) should not flag contradiction
smart_store.upsert_with_detection(Entity(
    "preference", "preference_python", "Likes Python",
    1, 0.8, ["python"],
))
r4 = smart_store.upsert_with_detection(Entity(
    "preference", "preference_python", "Loves Python and FastAPI",
    3, 0.9, ["python", "fastapi"],
))
assert r4["contradiction"] == False or r4["action"] == "update"
print(f"Preference update: {r4}")

print("\nAll SmartEntityStore tests passed!")

In [ ]:
#@title 🎧 Listen: Verify Todo2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_verify_todo2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 2

In [ ]:
# --- Test evaluate_retrieval_accuracy ---
test_cases = [
    {"query": "What is the budget?",
     "expected_key": "budget",
     "expected_value_keywords": ["5,000"],
     "category": "number"},
    {"query": "When is the deadline?",
     "expected_key": "deadline",
     "expected_value_keywords": ["march"],
     "category": "date"},
    {"query": "What is the user's name?",
     "expected_key": "user_name",
     "expected_value_keywords": ["raj"],
     "category": "name"},
    {"query": "What database are we using?",
     "expected_key": "tech_postgresql",
     "expected_value_keywords": ["postgresql"],
     "category": "technical"},
    {"query": "Does the user like Docker?",
     "expected_key": "tech_docker",
     "expected_value_keywords": ["docker"],
     "category": "technical"},
]

eval_result = evaluate_retrieval_accuracy(em, test_cases)

assert eval_result["total_cases"] == 5
assert 0 <= eval_result["accuracy"] <= 1
assert "number" in eval_result["per_category"]
print(f"Accuracy: {eval_result['correct']}/{eval_result['total_cases']} "
      f"({eval_result['accuracy']:.0%})")

for cat, stats in eval_result["per_category"].items():
    print(f"  {cat}: {stats['correct']}/{stats['total']} "
          f"({stats['accuracy']:.0%})")

if eval_result["failures"]:
    print(f"\nFailures: {eval_result['failures']}")

print("\nAll retrieval accuracy tests passed!")

In [ ]:
#@title 🎧 Listen: Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

<details>
<summary><b>Solution: SmartEntityStore.upsert_with_detection</b></summary>

In [ ]:
def upsert_with_detection(self, entity):
    result = {
        "action": "insert",
        "contradiction": False,
        "old_value": None,
        "new_value": entity.value,
        "key": entity.key,
    }

    if entity.key in self.entities:
        existing = self.entities[entity.key]
        result["old_value"] = existing.value
        result["action"] = "update"

        if (entity.key in self.single_valued
                and existing.value.lower() != entity.value.lower()):
            result["contradiction"] = True
            self.contradictions.append({
                "key": entity.key,
                "old_value": existing.value,
                "new_value": entity.value,
                "turn": entity.source_turn,
            })

        self.change_history[entity.key].append({
            "old_value": existing.value,
            "new_value": entity.value,
            "turn": entity.source_turn,
        })
        self.entities[entity.key] = entity
    else:
        self.entities[entity.key] = entity
        self.change_history[entity.key].append({
            "old_value": None,
            "new_value": entity.value,
            "turn": entity.source_turn,
        })

    self.history.append((result["action"], entity))
    return result

</details>

<details>
<summary><b>Solution: evaluate_retrieval_accuracy</b></summary>

In [ ]:
def evaluate_retrieval_accuracy(memory, test_cases):
    per_category = {}
    correct = 0
    failures = []

    for case in test_cases:
        cat = case["category"]
        if cat not in per_category:
            per_category[cat] = {"total": 0, "correct": 0, "accuracy": 0.0}
        per_category[cat]["total"] += 1

        results = memory.store.retrieve_relevant(case["query"], top_k=5)
        found = False
        for entity, score in results:
            if entity.key == case["expected_key"]:
                val_lower = entity.value.lower()
                if all(kw.lower() in val_lower for kw in case["expected_value_keywords"]):
                    found = True
                    break

        if found:
            correct += 1
            per_category[cat]["correct"] += 1
        else:
            failures.append(case["query"])

    for cat in per_category:
        t = per_category[cat]["total"]
        c = per_category[cat]["correct"]
        per_category[cat]["accuracy"] = c / t if t > 0 else 0

    return {
        "total_cases": len(test_cases),
        "correct": correct,
        "accuracy": correct / len(test_cases) if test_cases else 0,
        "per_category": per_category,
        "failures": failures,
    }

</details>

In [ ]:
#@title 🎧 Listen: Load Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_load_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Load solutions ---
class SmartEntityStore(EntityStore):
    def __init__(self):
        super().__init__()
        self.contradictions = []
        self.change_history = defaultdict(list)
        self.single_valued = {
            "budget", "deadline", "user_name",
            "count_people", "count_members",
        }

    def upsert_with_detection(self, entity):
        result = {
            "action": "insert", "contradiction": False,
            "old_value": None, "new_value": entity.value,
            "key": entity.key,
        }
        if entity.key in self.entities:
            existing = self.entities[entity.key]
            result["old_value"] = existing.value
            result["action"] = "update"
            if (entity.key in self.single_valued
                    and existing.value.lower() != entity.value.lower()):
                result["contradiction"] = True
                self.contradictions.append({
                    "key": entity.key,
                    "old_value": existing.value,
                    "new_value": entity.value,
                    "turn": entity.source_turn,
                })
            self.change_history[entity.key].append({
                "old_value": existing.value,
                "new_value": entity.value,
                "turn": entity.source_turn,
            })
            self.entities[entity.key] = entity
        else:
            self.entities[entity.key] = entity
            self.change_history[entity.key].append({
                "old_value": None, "new_value": entity.value,
                "turn": entity.source_turn,
            })
        self.history.append((result["action"], entity))
        return result

def evaluate_retrieval_accuracy(memory, test_cases):
    per_category = {}
    correct = 0
    failures = []
    for case in test_cases:
        cat = case["category"]
        if cat not in per_category:
            per_category[cat] = {"total": 0, "correct": 0, "accuracy": 0.0}
        per_category[cat]["total"] += 1
        results = memory.store.retrieve_relevant(case["query"], top_k=5)
        found = False
        for entity, score in results:
            if entity.key == case["expected_key"]:
                val_lower = entity.value.lower()
                if all(kw.lower() in val_lower for kw in case["expected_value_keywords"]):
                    found = True
                    break
        if found:
            correct += 1
            per_category[cat]["correct"] += 1
        else:
            failures.append(case["query"])
    for cat in per_category:
        t = per_category[cat]["total"]
        c = per_category[cat]["correct"]
        per_category[cat]["accuracy"] = c / t if t > 0 else 0
    return {
        "total_cases": len(test_cases),
        "correct": correct,
        "accuracy": correct / len(test_cases) if test_cases else 0,
        "per_category": per_category,
        "failures": failures,
    }

print("Solutions loaded!")

In [ ]:
#@title 🎧 Listen: Comparison Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_comparison_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Let us compare entity memory against summary memory on the same conversation, specifically on their ability to answer factual questions.

In [ ]:
# We already have em (EntityMemory) populated from earlier
# Now let us build a summary of the same conversation for comparison

class SimpleSummarizer:
    def summarize(self, text, max_length=200):
        entities = set()
        for word in text.split():
            cleaned = word.strip(".,!?\"'()[]")
            if cleaned and cleaned[0].isupper() and len(cleaned) > 1:
                if cleaned not in ("The", "This", "That", "I", "User", "Assistant"):
                    entities.add(cleaned)
        facts = []
        for match in re.finditer(r'\$[\d,]+', text):
            facts.append(f"Budget: {match.group()}")
        for match in re.finditer(r'((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2})', text):
            facts.append(f"Date: {match.group()}")
        parts = []
        if entities:
            parts.append(f"People/entities: {', '.join(sorted(entities)[:8])}")
        if facts:
            parts.append("Facts: " + "; ".join(facts[:5]))
        return ". ".join(parts) if parts else "General technical discussion."

summary_text = "\n".join(
    f"{m['role']}: {m['content']}"
    for m in em.all_messages
)
summarizer = SimpleSummarizer()
summary = summarizer.summarize(summary_text)
summary_tokens = count_tokens(summary)

print("=" * 60)
print("ENTITY MEMORY vs SUMMARY MEMORY")
print("=" * 60)
print(f"\nEntity store: {em.store.size} entities, "
      f"{em.store.total_tokens} tokens")
print(f"Summary: {summary_tokens} tokens")
print(f"\n--- Entity Store Contents ---")
for e in em.store.get_all():
    print(f"  {e.key}: {e.value}")
print(f"\n--- Summary ---")
print(f"  {summary}")

In [ ]:
#@title 🎧 Listen: Fact Retrieval Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_fact_retrieval_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Head-to-head fact retrieval comparison
fact_queries = [
    ("What is the budget?", "$5,000"),
    ("When is the deadline?", "March 15"),
    ("What is the user's name?", "Raj"),
    ("How many people on the team?", "4"),
    ("What does Sarah do?", "frontend"),
    ("What database are we using?", "PostgreSQL"),
    ("What does the user hate?", "meetings"),
]

print("\nFact Retrieval Comparison")
print("=" * 65)
print(f"{'Query':<30} {'Entity':>10} {'Summary':>10}")
print("-" * 52)

entity_hits = 0
summary_hits = 0

for query, expected in fact_queries:
    # Entity retrieval
    entity_results = em.store.retrieve_relevant(query, top_k=3)
    entity_text = " ".join(
        e.value.lower() for e, _ in entity_results
    )
    entity_found = expected.lower() in entity_text

    # Summary check
    summary_found = expected.lower() in summary.lower()

    entity_hits += entity_found
    summary_hits += summary_found

    e_mark = "HIT" if entity_found else "MISS"
    s_mark = "HIT" if summary_found else "MISS"
    print(f"  {query:<28} {e_mark:>10} {s_mark:>10}")

print(f"\n{'TOTAL':>30} {entity_hits}/{len(fact_queries):>7} "
      f"{summary_hits}/{len(fact_queries):>7}")
print(f"{'Accuracy':>30} {entity_hits/len(fact_queries):>7.0%} "
      f"{summary_hits/len(fact_queries):>7.0%}")

In [ ]:
#@title 🎧 Listen: Viz1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_viz1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

### Visualization 1: Entity Growth Over Conversation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Entity count over time
ax1 = axes[0]
msg_indices = list(range(1, len(em.entity_count_history) + 1))
ax1.plot(msg_indices, em.entity_count_history,
         color="#3B82F6", linewidth=2.5, marker="o",
         markersize=4)
ax1.fill_between(msg_indices, em.entity_count_history,
                 alpha=0.2, color="#3B82F6")
ax1.set_xlabel("Message Number", fontsize=12)
ax1.set_ylabel("Entities in Store", fontsize=12)
ax1.set_title("Entity Store Growth Over Conversation",
              fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)

# Right: Entity categories breakdown
ax2 = axes[1]
category_counts = defaultdict(int)
for e in em.store.get_all():
    category_counts[e.category] += 1

cats = list(category_counts.keys())
counts = [category_counts[c] for c in cats]
colors = ["#3B82F6", "#10B981", "#F59E0B",
          "#EF4444", "#8B5CF6", "#EC4899"]

bars = ax2.barh(cats, counts, color=colors[:len(cats)],
                edgecolor="white", height=0.5)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_width() + 0.1,
             bar.get_y() + bar.get_height()/2,
             str(count), va="center", fontsize=12,
             fontweight="bold")
ax2.set_xlabel("Number of Entities", fontsize=12)
ax2.set_title("Entity Distribution by Category",
              fontsize=14, fontweight="bold")
ax2.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("entity_growth.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Viz2 Token Cost
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_viz2_token_cost.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 2: Token Cost Comparison (Entity vs Summary vs Buffer)

In [ ]:
# Simulate costs for a 30-turn conversation
n_turns = 30
avg_tokens = 200

buffer_costs = [i * avg_tokens for i in range(1, n_turns + 1)]
window_costs = [min(i, 5) * avg_tokens for i in range(1, n_turns + 1)]
summary_costs = [300 + min(i, 5) * avg_tokens for i in range(1, n_turns + 1)]

# Entity memory: grows slowly as new entities are found
entity_costs = []
for i in range(1, n_turns + 1):
    # Entities grow sublinearly (fewer new entities over time)
    entity_tokens = min(150 + i * 15, 500)
    recent_tokens = min(i, 5) * avg_tokens
    entity_costs.append(entity_tokens + recent_tokens)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(1, n_turns + 1), buffer_costs,
        color="#EF4444", linewidth=2.5,
        label="Buffer (full)")
ax.plot(range(1, n_turns + 1), summary_costs,
        color="#F59E0B", linewidth=2.5,
        linestyle="--", label="Summary (K=5)")
ax.plot(range(1, n_turns + 1), entity_costs,
        color="#3B82F6", linewidth=2.5,
        linestyle="-.", label="Entity (K=5)")
ax.plot(range(1, n_turns + 1), window_costs,
        color="#10B981", linewidth=2.5,
        linestyle=":", label="Window (K=5)")

ax.axhline(y=8192, color="gray", linestyle=":",
           alpha=0.4, label="8K limit")

ax.set_xlabel("Conversation Turn", fontsize=12)
ax.set_ylabel("Context Tokens Sent to LLM", fontsize=12)
ax.set_title("Token Cost Comparison: Four Memory Strategies",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("entity_cost_comparison.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("Entity memory stays bounded like summary memory,")
print("but retrieves only relevant entities for each query.")

In [ ]:
#@title 🎧 Listen: Viz3 Accuracy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_viz3_accuracy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 3: Retrieval Accuracy Heatmap

In [ ]:
# Run the full evaluation
eval_result = evaluate_retrieval_accuracy(em, test_cases)

# Visualize accuracy per category
categories = list(eval_result["per_category"].keys())
accuracies = [
    eval_result["per_category"][c]["accuracy"]
    for c in categories
]

fig, ax = plt.subplots(figsize=(10, 5))

colors_bar = [
    "#10B981" if a >= 0.8 else "#F59E0B" if a >= 0.5
    else "#EF4444"
    for a in accuracies
]
bars = ax.bar(categories, accuracies, color=colors_bar,
              edgecolor="white", width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f"{acc:.0%}", ha="center", fontsize=12,
            fontweight="bold")

ax.set_ylim(0, 1.15)
ax.set_ylabel("Retrieval Accuracy", fontsize=12)
ax.set_title("Entity Retrieval Accuracy by Category",
             fontsize=14, fontweight="bold")
ax.axhline(y=0.8, color="gray", linestyle="--",
           alpha=0.4, label="80% threshold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("retrieval_accuracy.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"\nOverall accuracy: {eval_result['accuracy']:.0%}")

In [ ]:
#@title 🎧 Listen: Final Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_final_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us run a complete demonstration showing entity memory handling fact-specific queries across a long conversation.

In [ ]:
print("=" * 60)
print("  FINAL DEMO: ENTITY MEMORY FOR PRECISE FACT RECALL")
print("=" * 60)

# Simulate a new session where user asks about old facts
queries = [
    "What was the budget I mentioned?",
    "Who is on the team and what do they do?",
    "What is the project deadline?",
    "What tech stack are we using?",
    "Do I have any preferences you should know about?",
]

print("\n--- New Session: Retrieving Facts from Memory ---\n")

for query in queries:
    print(f"User: \"{query}\"")

    # Retrieve relevant entities
    relevant = em.store.retrieve_relevant(query, top_k=5)

    if relevant:
        print("  Retrieved facts:")
        for entity, score in relevant:
            print(f"    [{score:.2f}] {entity.key}: {entity.value}")

        # Format as assistant would use it
        context = em.store.format_for_prompt(
            [e for e, _ in relevant]
        )
        print(f"\n  Agent response (grounded in entities):")
        if "budget" in query.lower():
            budget_e = next(
                (e for e, _ in relevant if "budget" in e.key),
                None
            )
            if budget_e:
                print(f"    \"Your budget is {budget_e.value}.\"")
        elif "team" in query.lower():
            team_es = [
                e for e, _ in relevant
                if e.category == "relationship"
            ]
            if team_es:
                members = ", ".join(e.value for e in team_es)
                print(f"    \"{members}.\"")
        elif "deadline" in query.lower():
            date_e = next(
                (e for e, _ in relevant if "deadline" in e.key),
                None
            )
            if date_e:
                print(f"    \"The deadline is {date_e.value}.\"")
        elif "tech" in query.lower():
            tech_es = [
                e for e, _ in relevant
                if e.category == "technical"
            ]
            if tech_es:
                stack = ", ".join(e.value for e in tech_es)
                print(f"    \"Your stack includes: {stack}.\"")
        elif "preference" in query.lower():
            pref_es = [
                e for e, _ in relevant
                if e.category == "preference"
            ]
            if pref_es:
                prefs = "; ".join(e.value for e in pref_es)
                print(f"    \"Your preferences: {prefs}.\"")
    else:
        print("  No relevant entities found.")

    print()

# Final statistics
stats = em.get_stats()
print("=" * 60)
print(f"Entity memory: {stats['entities_stored']} facts stored")
print(f"Token cost: {stats['entity_tokens']} tokens for entities")
print(f"Every specific fact is preserved exactly as stated —")
print(f"no compression, no information loss on key details.")
print("=" * 60)

In [ ]:
#@title 🎧 Listen: Reflection Next Steps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_reflection_next_steps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What Did We Build?

1. **Entity extractor** — identifies names, numbers, dates, preferences, and technical facts from conversation text using pattern matching.
2. **Entity store** — structured key-value storage with deduplication, contradiction detection, and relevance-based retrieval.
3. **Entity memory system** — integrates extraction, storage, and prompt building into a seamless pipeline.
4. **Evaluation framework** — measures retrieval accuracy per fact category.

### The Key Tradeoff

Entity memory excels at **specific fact recall** but struggles with **general context**. It knows that the budget is $5,000 but cannot describe the overall direction of the conversation. Summary memory is the opposite — great at general context, poor at specific facts.

This suggests an obvious next step: combine them. Use entity memory for precise facts and summary memory (or vector retrieval) for general context.

### Questions to Think About

1. Our extractor is rule-based. In production, you would use an LLM with a prompt like "Extract all key entities from this message as JSON." What patterns would an LLM catch that our rules miss?

2. How would you handle entity **merging**? If the user says "PostgreSQL" in one turn and "Postgres" in another, should these be the same entity?

3. What if the number of entities grows very large (hundreds)? How would you decide which entities to keep and which to prune?

4. Entity memory uses exact keyword matching for retrieval. What would change if you embedded entity descriptions and used cosine similarity?

5. Can you think of a scenario where entity memory would give a *wrong* answer? (Hint: what if the user says "My friend's budget is $10,000" — would the system incorrectly store this as the user's budget?)

### What Comes Next

Entity memory gives us precision but not semantic understanding. In the next notebook, we will build **vector-backed memory** — using embeddings and FAISS to retrieve memories by semantic similarity. This is what allows the system to answer "How much can I spend?" even when the stored fact says "budget is $5,000" — because the embeddings capture that "spend" and "budget" are semantically related.

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("Notebook complete!")
print()
print("Key concepts:")
print("  Entity extraction: pattern-based fact identification")
print("  Structured storage: key-value with categories")
print("  Contradiction detection: single-valued key conflicts")
print("  Retrieval accuracy: precision/recall on fact queries")
print()
print("Entity memory = precision. Summary memory = coverage.")
print("Next up: Vector Memory — semantic retrieval with embeddings.")